<a href="https://colab.research.google.com/github/Alice-Ferri/data-intensive-project-2026/blob/main/data-intensive-project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Determinare la categoria della malattia della cellula.

Programmazione di data intensive a.a. 2025/2026

Alice Ferri, alice.ferri8@studio.unibo.it

Davide rossi, davide.rossi47@studio.unibo.it

### Parte 1 - Descrizione del contesto del problema

Il dataset di riferimento è [rilevamento di anomalie di cellule del sangue](https://www.kaggle.com/datasets/alitaqishah/blood-cell-anomaly-detection-2025/data) presente in Kaggle.

Il dataset contiene le informazioni di cellule del sangue sane e malate.
Tali dati permettono di suddividere le cellule in 7 categorie, 5 di queste classificate come anomale e 2 come normali.

L'obbiettivo del progetto è sviluppare un classificatore che sia in grando di determinare la categoria della cellula.

### Caricamento librerie

Installiamo nel kernel la libreria per importare il dataset da kagglehub

In [14]:
pip install kagglehub[pandas-datasets]

Note: you may need to restart the kernel to use updated packages.


Importiamo le librerie che utilizzeremo

In [9]:
import kagglehub
import numpy as np
import pandas as pd
from kagglehub import KaggleDatasetAdapter

### Caricamento dei dati e preprocessing

Carichiamo il dataset in un pandas dataframe da kagglehub

In [10]:
# nome del file del dataset
file_path = "blood_cell_anomaly_detection.csv"

data_raw = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "alitaqishah/blood-cell-anomaly-detection-2025",
  file_path
)

data_raw.tail()

,cell_id,cell_type,anomaly_label,disease_category,cell_diameter_um,nucleus_area_pct,chromatin_density,cytoplasm_ratio,circularity,eccentricity,...,mcv_fl,mchc_g_dl,dataset_source,staining_protocol,microscope_model,magnification_x,image_resolution_px,cytodiffusion_anomaly_score,cytodiffusion_classification_confidence,labeller_confidence_score
5875,CELL_003773,Platelet,0,Normal_Platelet,2.30,0.0,0.000,1.000,0.699,0.536,...,81.4,34.2,CytoData,May_Grunwald_Giemsa,Olympus_BX51,40,360,0.1178,1.0000,0.8849
5876,CELL_005192,Target_Cell,1,Anemia,8.80,0.0,0.000,1.000,0.831,0.199,...,88.1,34.4,Raabin_WBC,Giemsa,Zeiss_Axio,40,224,0.9034,0.9475,0.5928
5877,CELL_005227,Target_Cell,1,Anemia,9.57,0.0,0.000,1.000,0.900,0.320,...,86.7,34.0,PBC_Dataset,Giemsa,Olympus_BX51,40,256,0.8493,0.6716,0.6381
5878,CELL_005391,Hypersegmented_Neutrophil,1,Infection,11.98,67.8,0.528,0.309,0.752,0.505,...,96.5,32.9,CytoData,Giemsa,Leica_DM2000,100,256,0.9004,0.6449,0.8801
5879,CELL_000861,Neutrophil,0,Normal_WBC,12.58,51.0,0.619,0.471,0.764,0.341,...,82.7,32.5,CytoData,Giemsa,Olympus_BX51,100,360,0.0854,1.0000,0.9553


Con il metodo info stampiamo le informazioni principali come numero di istanze, data type
per feature, e spazio in memoria

In [3]:
data_raw.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 5880 entries, 0 to 5879
Data columns (total 36 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   cell_id                                  5880 non-null   str    
 1   cell_type                                5880 non-null   str    
 2   anomaly_label                            5880 non-null   int64  
 3   disease_category                         5880 non-null   str    
 4   cell_diameter_um                         5880 non-null   float64
 5   nucleus_area_pct                         5880 non-null   float64
 6   chromatin_density                        5880 non-null   float64
 7   cytoplasm_ratio                          5880 non-null   float64
 8   circularity                              5880 non-null   float64
 9   eccentricity                             5880 non-null   float64
 10  granularity_score                        5880 non-null   fl

### Significato delle features
Il dataset contiene 36 features, di seguito sono riportate raggruppate per tipologia:

**Morphology** — diametro, circolarità, eccentricità, lobularità, granularità, area del nucleo, densità della cromatina

**Color** — valori RGB medi, intensità della colorazione

**Clinical CBC** — dati ricavati dagli esami del sangue, come quantità globuli bianchi e rossi, piastrine, emoglobina etc.

**Acquisition** — dati riguardanti l'immagine al microscopio, come il modello, la risoluzione e grado di ingrandimento

**AI Scores** — dati legati al modello CytoDiffusion che risolve la stessa tipologia di problema, come confidenza di anomalia della cellula e di suddivisione del tipo della cellula. Troviamo anche il valore di sicurezza della stima del tipo di cellula di un medico esperto

La variabile che tenteremo di predirre è **disease_category**. Le label possono essere:
- __Normal_WBC__, globuli bianchi normali
- __Normal_RBC__, globuli rossi normali
- __Leukemia__
- __Anemia__
- __Sickle_Cell_Anemia__, anemia falciforme
- __Infection__
- __Artefact__


### Scrematura dei dati

In [6]:
ds = data_raw.copy()

ds.drop(columns=['cell_id',
                 'dataset_source',
                 'cytodiffusion_anomaly_score',
                 'cytodiffusion_classification_confidence',
                 'labeller_confidence_score'], inplace=True)

In [8]:
ds.head(10)

,cell_type,anomaly_label,disease_category,cell_diameter_um,nucleus_area_pct,chromatin_density,cytoplasm_ratio,circularity,eccentricity,granularity_score,...,rbc_count_millions_per_ul,hemoglobin_g_dl,hematocrit_pct,platelet_count_per_ul,mcv_fl,mchc_g_dl,staining_protocol,microscope_model,magnification_x,image_resolution_px
0,Hypersegmented_Neutrophil,1,Infection,15.18,58.8,0.542,0.301,0.563,0.529,4.11,...,4.44,11.7,43.4,257383,85.5,31.4,Giemsa,Zeiss_Axio,100,224
1,Hypersegmented_Neutrophil,1,Infection,16.47,73.6,0.583,0.365,0.859,0.443,2.50,...,4.90,13.9,42.2,302274,92.5,35.0,Wright,Zeiss_Axio,100,224
2,Neutrophil,0,Normal_WBC,13.41,55.5,0.448,0.376,0.781,0.407,3.01,...,5.72,16.1,39.2,229996,76.3,33.0,Wright,Leica_DM2000,100,512
3,Normal_RBC,0,Normal_RBC,7.36,0.0,0.000,1.000,0.880,0.167,0.43,...,3.42,14.6,54.1,130720,92.3,32.5,Wright,Leica_DM2000,100,512
4,Normal_RBC,0,Normal_RBC,7.53,0.0,0.000,1.000,1.000,0.158,0.51,...,5.36,14.6,36.7,228652,83.9,33.4,Wright,Olympus_BX51,100,224
5,Reactive_Lymphocyte,1,Infection,14.97,57.8,0.730,0.188,0.777,0.351,1.66,...,2.70,14.9,37.2,130816,93.7,32.1,Giemsa,Leica_DM2000,100,224
6,Monocyte,0,Normal_WBC,15.63,63.3,0.503,0.571,0.559,0.312,1.79,...,4.01,14.7,39.6,168661,80.9,34.1,Giemsa,Olympus_BX51,60,360
7,Lymphocyte,0,Normal_WBC,9.15,78.3,0.790,0.130,0.820,0.045,0.70,...,4.04,10.9,38.4,389245,88.5,33.9,Giemsa,Olympus_BX51,60,512
8,Smudge_Cell,1,Artefact,10.10,92.9,0.685,0.050,0.288,0.761,0.15,...,5.07,13.8,43.6,217608,80.6,32.0,May_Grunwald_Giemsa,Leica_DM2000,100,512
9,Monocyte,0,Normal_WBC,16.97,64.1,0.454,0.449,0.785,0.575,2.08,...,4.36,9.9,43.4,248113,66.3,32.8,Wright,Olympus_BX51,100,360
